# Methods Summary Document

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
from pathlib import Path

from docx import Document
from docx.enum.section import WD_SECTION_START
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor


OUT = Path("/Users/lu_nanxi/CASA/Dissertation_Data/dissertation_methods_vivacity_exploratory_analysis.docx")


def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def set_cell_margins(cell, top=80, start=120, bottom=80, end=120):
    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for m, v in [("top", top), ("start", start), ("bottom", bottom), ("end", end)]:
        node = tc_mar.find(qn(f"w:{m}"))
        if node is None:
            node = OxmlElement(f"w:{m}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")


def set_table_borders(table):
    tbl = table._tbl
    tbl_pr = tbl.tblPr
    borders = tbl_pr.first_child_found_in("w:tblBorders")
    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tbl_pr.append(borders)
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        tag = f"w:{edge}"
        elem = borders.find(qn(tag))
        if elem is None:
            elem = OxmlElement(tag)
            borders.append(elem)
        elem.set(qn("w:val"), "single")
        elem.set(qn("w:sz"), "4")
        elem.set(qn("w:space"), "0")
        elem.set(qn("w:color"), "DADCE0")


def add_para(doc, text, style=None):
    p = doc.add_paragraph(style=style)
    p.paragraph_format.space_after = Pt(8)
    p.paragraph_format.line_spacing = 1.15
    run = p.add_run(text)
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.size = Pt(11)
    return p


def add_heading(doc, text, level=1):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(20 if level == 1 else 18 if level == 2 else 16)
    p.paragraph_format.space_after = Pt(6 if level in (1, 2) else 4)
    run = p.add_run(text)
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.color.rgb = RGBColor(0, 0, 0) if level < 3 else RGBColor(67, 67, 67)
    run.font.size = Pt(20 if level == 1 else 16 if level == 2 else 14)
    return p


def add_bullets(doc, items):
    for item in items:
        p = doc.add_paragraph(style="List Bullet")
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        run = p.add_run(item)
        run.font.name = "Arial"
        run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
        run.font.size = Pt(11)


def build_doc():
    doc = Document()

    section = doc.sections[0]
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)
    section.header_distance = Inches(0.492)
    section.footer_distance = Inches(0.492)

    styles = doc.styles
    styles["Normal"].font.name = "Arial"
    styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    styles["Normal"].font.size = Pt(11)

    title = doc.add_paragraph()
    title.paragraph_format.space_before = Pt(0)
    title.paragraph_format.space_after = Pt(3)
    run = title.add_run("Methods Draft: Vivacity Active Travel Analysis")
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.size = Pt(26)
    run.font.color.rgb = RGBColor(0, 0, 0)

    subtitle = doc.add_paragraph()
    subtitle.paragraph_format.space_after = Pt(12)
    subtitle_run = subtitle.add_run("Exploratory matched-control time-series approach for CASA dissertation")
    subtitle_run.font.name = "Arial"
    subtitle_run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    subtitle_run.font.size = Pt(11)
    subtitle_run.font.color.rgb = RGBColor(85, 85, 85)

    add_heading(doc, "Overview", 1)
    add_para(
        doc,
        "This study uses automated Vivacity countline sensor data to examine walking and cycling trajectories around selected Liverpool City Region active travel schemes. The analysis focuses on whether schemes located in different socio-economic contexts show different post-intervention trajectories, and whether these trajectories differ from matched non-intervention sensor locations where suitable comparison data are available.",
    )

    add_heading(doc, "Study Sites and Treated Schemes", 1)
    add_para(
        doc,
        "The treated schemes were identified from Liverpool City Region Combined Authority scheme records and Vivacity sensor metadata. Five scheme groups were initially considered: 12b Cronton Road / Sandy Lane, 12d Aigburth Road, 12e Park Road / Path off Park Road, 12f Leasowe / Wallasey corridor, and 13 Astmoor / Warrington Road.",
    )
    add_para(
        doc,
        "For each scheme, relevant Vivacity countlines were cleaned to daily counts by summing hourly directional records across the day. The main outcomes were daily pedestrian counts, cyclist counts, and combined active travel counts.",
    )

    add_heading(doc, "Data Cleaning and Full-Day Cutoff", 1)
    add_para(
        doc,
        "To avoid using incomplete recent observations, all Vivacity daily datasets were filtered to include only full days up to 26 May 2026. For each countline, unreliable early sensor periods were identified using data availability and observed-flow criteria.",
    )
    add_para(
        doc,
        "A day was treated as usable only where data availability was present, minimum daily availability was at least 80%, no data error was recorded, and the row occurred on or after the countline’s first reliable date. The first reliable date was defined as the first day with positive active travel flow and at least 14 reliable days within the following 21-day window. Rows before this date were treated as structural sensor absence rather than genuine zero flow.",
    )

    add_heading(doc, "Contextual and Matched-Control Data", 1)
    add_para(
        doc,
        "Matched control countlines were selected using LSOA 2021 contextual similarity. Treated sensor locations were spatially joined to LSOA 2021 boundaries and linked to socio-economic context variables, including Index of Multiple Deprivation deciles, income and employment deprivation, population density, car availability, and Census 2021 walking and cycling commute shares.",
    )
    add_para(
        doc,
        "Candidate control LSOAs were ranked by similarity to treated LSOAs, with Vivacity countlines located in high-ranking non-intervention areas shortlisted for download and cleaning. These candidate controls were then subjected to the same data-quality gate as treated sensors.",
    )

    add_heading(doc, "Integrity Gate and Inclusion Rules", 1)
    add_para(
        doc,
        "A formal inclusion gate was applied before modelling. Controls were included in the exploratory causal comparison only if their first reliable date preceded the confirmed intervention date, they had at least 90 usable pre-intervention days, and at least 180 usable post-intervention days. A stricter seasonal threshold of 365 usable pre- and post-intervention days was also recorded, but only two controls met this standard.",
    )
    add_para(
        doc,
        "Controls failing the pragmatic threshold were retained for descriptive comparison only and excluded from the main treated-control models. All control sites still require manual verification to confirm that they are not themselves affected by active travel interventions.",
    )

    add_heading(doc, "Scheme Inclusion Status", 2)
    table = doc.add_table(rows=1, cols=4)
    table.autofit = False
    table.columns[0].width = Inches(0.85)
    table.columns[1].width = Inches(1.45)
    table.columns[2].width = Inches(1.7)
    table.columns[3].width = Inches(2.45)
    set_table_borders(table)
    headers = ["Scheme", "Model date", "Analysis status", "Reason"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.LEFT
        r = p.add_run(header)
        r.bold = True
        r.font.name = "Arial"
        r._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
        r.font.size = Pt(10)

    rows = [
        ("12b", "2023-03-01", "Descriptive only", "No quality-passing pre-intervention treated baseline is available under the current gate."),
        ("12d", "2023-03-01", "Exploratory treated-control model", "Treated sensors and a small set of pragmatic matched controls have sufficient pre/post data."),
        ("12e", "2023-03-01", "Descriptive only", "No quality-passing pre-intervention treated baseline is available under the current gate."),
        ("12f", "2023-06-01", "Exploratory treated-control model", "Treated sensors and pragmatic matched controls have sufficient usable pre/post data."),
        ("13", "2024-03-01", "Exploratory treated-control model", "Treated sensors and the strongest matched-control pool are available; two controls meet the stricter seasonal threshold."),
    ]
    for row in rows:
        cells = table.add_row().cells
        for i, text in enumerate(row):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            r = p.add_run(text)
            r.font.name = "Arial"
            r._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
            r.font.size = Pt(10)

    add_heading(doc, "Exploratory Modelling Strategy", 1)
    add_para(
        doc,
        "For 12d, 12f, and 13, weekly countline-level models were estimated separately by scheme and outcome. Daily records were aggregated to weekly counts per observed countline-day to reduce short-term noise and account for occasional missing days. Models were fitted for active travel, pedestrian counts, and cyclist counts.",
    )
    add_para(
        doc,
        "The exploratory model specification estimated relative treated-control changes after the intervention month, including a post-intervention level term and a post-intervention slope term, while controlling for underlying time trend, seasonality, and countline fixed effects. Standard errors were clustered by countline.",
    )

    add_heading(doc, "Interpretation and Limitations", 1)
    add_para(
        doc,
        "The model remains exploratory rather than a definitive causal impact estimate because controls are limited and still require manual verification. Confirmed intervention dates are encoded in the current workflow.",
    )
    add_para(
        doc,
        "Results should therefore be interpreted as evidence of relative trajectory differences rather than conclusive scheme effects. In dissertation writing, terms such as ‘relative trajectory’, ‘exploratory treated-control comparison’, and ‘evidence consistent with’ are preferable to stronger causal language.",
    )

    add_heading(doc, "Reproducibility Notes", 1)
    add_bullets(
        doc,
        [
            "Full-day cutoff dataset: Vivacity_full_day_cutoff_20260526.",
            "Integrity gate outputs: Vivacity_full_day_cutoff_20260526/integrity_gate.",
            "Modelling-ready tables: Vivacity_full_day_cutoff_20260526/modelling_ready/tables.",
            "Exploratory model outputs: Vivacity_full_day_cutoff_20260526/modelling_ready/models.",
        ],
    )

    doc.save(OUT)


if __name__ == "__main__":
    build_doc()
    print(OUT)
